In [13]:
import copy
from collections import deque

# Performance tracking counters
backtrack_calls = 0
failure_count = 0

def load_sudoku(file_path):
    """Reads the Sudoku grid from a text file."""
    grid = []
    try:
        with open(file_path, 'r') as f:
            for line in f:
                line = line.strip()
                if line:
                    grid.append([int(char) for char in line])
        return grid
    except Exception as e:
        print(f"Error reading file: {e}")
        return None

def display_grid(grid):
    """Prints the 9x9 Sudoku board clearly."""
    for row in grid:
        print(" ".join(str(num) for num in row))

def get_related_cells(r, c):
    """Identifies cells in the same row, column, and 3x3 subgrid."""
    related = set()
    for i in range(9):
        related.add((r, i))  # Row
        related.add((i, c))  # Column
    
    # 3x3 Box coordinates
    box_row, box_col = 3 * (r // 3), 3 * (c // 3)
    for i in range(box_row, box_row + 3):
        for j in range(box_col, box_col + 3):
            related.add((i, j))
            
    related.remove((r, c))
    return related

def initialize_domains(grid):
    """Initializes possible values for each cell based on initial hints."""
    domains = {}
    for r in range(9):
        for c in range(9):
            if grid[r][c] != 0:
                domains[(r, c)] = [grid[r][c]]
            else:
                domains[(r, c)] = list(range(1, 10))
    return domains

def enforce_constraint(domains, cell_i, cell_j):
    """Removes values from cell_i's domain that conflict with cell_j's fixed value."""
    changed = False
    if len(domains[cell_j]) == 1:
        fixed_val = domains[cell_j][0]
        if fixed_val in domains[cell_i]:
            domains[cell_i].remove(fixed_val)
            changed = True
    return changed

def run_ac3(domains):
    """Maintains arc consistency across the board to prune impossible values."""
    queue = deque()
    for cell_i in domains:
        for cell_j in get_related_cells(*cell_i):
            queue.append((cell_i, cell_j))
            
    while queue:
        cell_i, cell_j = queue.popleft()
        if enforce_constraint(domains, cell_i, cell_j):
            if not domains[cell_i]:
                return False
            for neighbor in get_related_cells(*cell_i):
                if neighbor != cell_j:
                    queue.append((neighbor, cell_i))
    return True

def validate_move(grid, r, c, val):
    """Checks if placing 'val' at (r, c) satisfies Sudoku rules."""
    for i in range(9):
        if grid[r][i] == val or grid[i][c] == val:
            return False
            
    br, bc = 3 * (r // 3), 3 * (c // 3)
    for i in range(br, br + 3):
        for j in range(bc, bc + 3):
            if grid[i][j] == val:
                return False
    return True

def apply_forward_checking(domains, cell, val):
    """Reduces neighbor domains after assigning a value to a cell."""
    updated_domains = copy.deepcopy(domains)
    updated_domains[cell] = [val]
    for neighbor in get_related_cells(*cell):
        if val in updated_domains[neighbor]:
            updated_domains[neighbor].remove(val)
            if not updated_domains[neighbor]:
                return None
    return updated_domains

def find_best_cell(grid, domains):
    """MRV Heuristic: Picks the empty cell with the smallest domain size."""
    best_cell = None
    min_size = 11
    for r in range(9):
        for c in range(9):
            if grid[r][c] == 0:
                size = len(domains[(r, c)])
                if size < min_size:
                    min_size = size
                    best_cell = (r, c)
    return best_cell

def solve_recursive(grid, domains):
    """Backtracking algorithm optimized with forward checking."""
    global backtrack_calls, failure_count
    backtrack_calls += 1
    
    target = find_best_cell(grid, domains)
    if target is None:
        return grid # Solved
        
    r, c = target
    for val in domains[target]:
        if validate_move(grid, r, c, val):
            new_grid = [row[:] for row in grid]
            new_grid[r][c] = val
            
            new_domains = apply_forward_checking(domains, target, val)
            if new_domains is not None:
                result = solve_recursive(new_grid, new_domains)
                if result is not None:
                    return result
                    
    failure_count += 1
    return None

def run_solver(file_name):
    """Handles the solving process for a specific file."""
    global backtrack_calls, failure_count
    backtrack_calls = 0
    failure_count = 0
    
    grid = load_sudoku(file_name)
    if not grid:
        return

    domains = initialize_domains(grid)
    
    print("-" * 35)
    print(f"File: {file_name}")
    
    if run_ac3(domains):
        # Sync fixed values to grid
        for cell, values in domains.items():
            if len(values) == 1:
                grid[cell[0]][cell[1]] = values[0]
                
        solution = solve_recursive(grid, domains)
        
        if solution:
            print("\nSolved Board:")
            display_grid(solution)
        else:
            print("No solution found.")
            
        print(f"\nBacktrack Calls: {backtrack_calls}")
        print(f"Failures: {failure_count}")
    else:
        print("Initial board is invalid.")

if __name__ == "__main__":
    files = ["easy.txt", "medium.txt", "hard.txt", "veryhard.txt"]
    for f in files:
        run_solver(f)

-----------------------------------
File: easy.txt

Solved Board:
7 8 4 9 3 2 1 5 6
6 1 9 4 8 5 3 2 7
2 3 5 1 7 6 4 8 9
5 7 8 2 6 1 9 3 4
3 4 1 8 9 7 5 6 2
9 2 6 5 4 3 8 7 1
4 5 3 7 2 9 6 1 8
8 6 2 3 1 4 7 9 5
1 9 7 6 5 8 2 4 3

Backtrack Calls: 1
Failures: 0
-----------------------------------
File: medium.txt

Solved Board:
4 3 5 2 6 9 7 8 1
6 8 2 5 7 1 4 9 3
1 9 7 8 3 4 5 6 2
8 2 6 1 9 5 3 4 7
3 7 4 6 8 2 9 1 5
9 5 1 7 4 3 6 2 8
5 1 9 3 2 6 8 7 4
2 4 8 9 5 7 1 3 6
7 6 3 4 1 8 2 5 9

Backtrack Calls: 1
Failures: 0
-----------------------------------
File: hard.txt

Solved Board:
3 7 4 1 6 2 9 5 8
6 8 5 3 7 9 2 1 4
2 1 9 5 8 4 6 7 3
5 6 8 7 2 3 4 9 1
9 2 1 6 4 5 3 8 7
7 4 3 9 1 8 5 6 2
8 9 6 4 3 1 7 2 5
1 3 7 2 5 6 8 4 9
4 5 2 8 9 7 1 3 6

Backtrack Calls: 434
Failures: 368
-----------------------------------
File: veryhard.txt

Solved Board:
4 7 9 5 3 6 8 1 2
2 6 8 7 1 4 3 5 9
5 3 1 8 2 9 4 6 7
8 4 2 3 9 5 1 7 6
7 5 3 2 6 1 9 4 8
1 9 6 4 8 7 5 2 3
6 8 5 9 4 2 7 3 1
3 1 7 6 5 8 2 9 4
